<div class="alert alert-block alert-info">
<b>Number of points for this notebook:</b> 8
<br>
<b>Deadline:</b> November 11, 2025 (Tuesday) 23:59
<br>
<b>In the final submission, please only modify the cell with #YOUR CODE HERE
<br>
<b>Please only submit one .zip file that contains all files.
</div>

# Exercise 4. Sequence-to-sequence modeling with recurrent neural networks

The goals of this exercise are
* to get familiar with recurrent neural networks used for sequential data processing
* to get familiar with the sequence-to-sequence model for machine translation
* to learn PyTorch tools for batch processing of sequences with varying lengths
* to learn how to write a custom `DataLoader`

You may find it useful to look at this tutorial:
* [Translation with a Sequence to Sequence Network and Attention](https://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html)

<div class="alert alert-block alert-info">
Uncomment the cell below to install the required packages in your environment or colab.
<br>
<b>Comment the cell before submitting to the grader.</b>
</div>

In [1]:
#uncomment to install the package, comment before submission
#!pip install evaluate

In [55]:
skip_training = True  # Set this flag to True before validation and submission

In [ ]:
# During evaluation, this cell sets skip_training to True
# skip_training = True

In [3]:
import os
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
import evaluate

import tools
import tests


c:\Users\guzma\4 IMAT\PRIMER CUATRI\Machines Learning and Perception\Practica_5_RNNs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# When running on your own computer, you can specify the data directory by:
# data_dir = tools.select_data_dir('/your/local/data/directory')
data_dir = tools.select_data_dir()

The data directory is ../data


In [5]:
# Select the device for training (use GPU if you have one)
#device = torch.device('cuda:0')
device = torch.device('cpu')

In [6]:
if skip_training:
    # The models are always evaluated on CPU
    device = torch.device("cpu")

## Data

The dataset that we are going to use consists of pairs of sentences in French and English.

In [7]:
from data import TranslationDataset, MAX_LENGTH, SOS_token, EOS_token, Lang

trainset = TranslationDataset(data_dir, train=True)

* `TranslationDataset` supports indexing as required by `torch.utils.data.Dataset`.
* Sentences are tensors of maximum length `MAX_LENGTH`.
* Words in a (sentence) tensor are represented as an index (integer) in a language vocabulary.
* The string representation of a word from the source language can be obtained from index `i` with `dataset.input_lang.index2word[i]`.
* Similarly for the target language `dataset.output_lang.index2word[j]`.

Let us look at samples from that dataset.

In [8]:
src_sentence, tgt_sentence = trainset[np.random.choice(len(trainset))]
print('Source sentence: "%s"' % ' '.join(trainset.input_lang.index2word[i.item()] for i in src_sentence))
print('Sentence as tensor of word indices:')
print(src_sentence)

print('Target sentence: "%s"' % ' '.join(trainset.output_lang.index2word[i.item()] for i in tgt_sentence))
print('Sentence as tensor of word indices:')
print(tgt_sentence)

Source sentence: "c est une fille tres chouette . EOS"
Sentence as tensor of word indices:
tensor([ 145,   25,  299,  999,  121, 1596,    5,    1])
Target sentence: "she s a very nice girl . EOS"
Sentence as tensor of word indices:
tensor([ 75,  15,  42, 304, 124, 909,   4,   1])


In [9]:
print('Number of source-target pairs in the training set: ', len(trainset))

Number of source-target pairs in the training set:  8682


## Sequence-to-sequence model for machine translation

In this exercise, we are going to build a machine translation system which transforms a sentence in one language into a sentence in another one. The computational graph of the translation model is shown below:

<img src="seq2seq.png" width=900>

We are going to use a simplified model without the dotted connections.

## Custom DataLoader

We would like to train the sequence-to-sequence model using mini-batch training.
One difficulty of mini-batch training in this case is that sequences may have varying lengths and this has to be taken into account when building the computational graph. Luckily, PyTorch has tools to support batch processing of such sequences.
To use those tools, we need to write a custom data loader which puts sequences of varying lengths in the same tensor. We can customize the data loader by providing a custom `collate_fn` as explained [here](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader).

Our collate function:
- combines sequences from the source language in a single tensor with extra values (at the end) filled with `PADDING_VALUE=0`.
- combines sequences from the target language in a single tensor with extra values (at the end) filled with `PADDING_VALUE=0`.

**Important**:
- Later in the code (not in this `collate` function), we will convert source sequences to objects of class [`PackedSequence`](https://pytorch.org/docs/stable/nn.html?highlight=packedsequence#torch.nn.utils.rnn.PackedSequence) which can be processed by recurrent units such as `GRU` or `LSTM`. Conversion to `PackedSequence` objects requires sequences to be sorted by their lengths.
**Therefore, the returned source sequences should be sorted by length in a decreasing order.**
* The target sequences need not be sorted by their lengths because we have to keep the same order of sequences in the source and target tensors.

Your task is to implement the collate function.

In [10]:
PADDING_VALUE = 0

In [42]:
from torch.nn.utils.rnn import pad_sequence

def collate(list_of_samples):
    """Merges a list of samples to form a mini-batch.

    Args:
      list_of_samples is a list of tuples (src_seq, tgt_seq):
          src_seq is of shape (src_seq_length,)
          tgt_seq is of shape (tgt_seq_length,)

    Returns:
      src_seqs of shape (max_src_seq_length, batch_size): Tensor of padded source sequences.
          The sequences should be sorted by length in a decreasing order, that is src_seqs[:,0] should be
          the longest sequence, and src_seqs[:,-1] should be the shortest.
      src_seq_lengths: List of lengths of source sequences.
      tgt_seqs of shape (max_tgt_seq_length, batch_size): Tensor of padded target sequences.
    """
    # YOUR CODE HERE

    pairs = [(src, tgt, src.size(0)) for (src, tgt) in list_of_samples]
    pairs.sort(key=lambda x: x[2], reverse=True)  # ordenar por len(src) desc

    src_sorted = [p[0] for p in pairs]
    tgt_sorted = [p[1] for p in pairs]
    src_seq_lengths = [p[2] for p in pairs]

    src_seqs = pad_sequence(src_sorted, batch_first=False, padding_value=PADDING_VALUE)
    tgt_seqs = pad_sequence(tgt_sorted, batch_first=False, padding_value=PADDING_VALUE)
    return src_seqs, src_seq_lengths, tgt_seqs

    

In [43]:
def test_collate_shapes():
    pairs = [
        (torch.LongTensor([1, 2]), torch.LongTensor([3, 4, 5])),
        (torch.LongTensor([6, 7, 8]), torch.LongTensor([9, 10])),
    ]
    pad_src_seqs, src_seq_lengths, pad_tgt_seqs = collate(pairs)
    assert type(src_seq_lengths) == list, "src_seq_lengths should be a list."
    assert pad_src_seqs.shape == torch.Size([3, 2]), f"Bad pad_src_seqs.shape: {pad_src_seqs.shape}"
    assert pad_src_seqs.dtype == torch.long
    assert pad_tgt_seqs.shape == torch.Size([3, 2]), f"Bad pad_tgt_seqs.shape: {pad_tgt_seqs.shape}"
    assert pad_tgt_seqs.dtype == torch.long
    print('Success')

test_collate_shapes()

Success


In [ ]:
# This cell tests collate() function

In [44]:
# We create custom DataLoader using the implemented collate function
# We are going to process 64 sequences at the same time (batch_size=64)
trainloader = DataLoader(dataset=trainset, batch_size=64, shuffle=True, collate_fn=collate, pin_memory=True)

## Encoder

The encoder encodes a source sequence $(x_1, x_2, ..., x_T)$ into a single vector $h_T$ using the following recursion:
$$
  h_{t} = f(h_{t-1}, x_t) \qquad t = 1, \ldots, T
$$
where:
* intial state $h_0$ is often chosen arbitrarily (we choose it to be zero)
* function $f$ is defined by the type of the RNN cell (in our experiments, we will use [GRU](https://pytorch.org/docs/stable/nn.html#torch.nn.GRU))
* $x_t$ is a vector that represents the $t$-th word in the source sentence.

A common practice in natural language processing is to _learn_ the word representations $x_t$ (instead of, for example, using one-hot coded vectors). In PyTorch, this is supported by class [Embedding](https://pytorch.org/docs/stable/nn.html#torch.nn.Embedding) which we are going to use.

The computational graph of the encoder is shown below:

<img src="seq2seq_encoder.png" width=500>

Your task is to implement the `forward` function of the encoder. It should contain the following steps:
* Embed the words of the source sequences.
* Pack source sequences using [`pack_padded_sequence`](https://pytorch.org/docs/stable/nn.html?highlight=pack_padded_sequence#torch.nn.utils.rnn.pack_padded_sequence). This converts padded source sequences into an object that can be processed by PyTorch recurrent units such as `nn.GRU` or `nn.LSTM`.
* Apply GRU computations to packed sequences obtained in the previous step
* Convert packed sequence of GRU outputs into padded representation with [`pad_packed_sequence`](https://pytorch.org/docs/stable/nn.html?highlight=pad_packed_sequence#torch.nn.utils.rnn.pad_packed_sequence).

In [45]:
class Encoder(nn.Module):
    def __init__(self, src_dictionary_size, embed_size, hidden_size):
        """
        Args:
          src_dictionary_size: The number of words in the source dictionary.
          embed_size: The number of dimensions in the word embeddings.
          hidden_size: The number of features in the hidden state of GRU.
        """
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(src_dictionary_size, embed_size)
        self.gru = nn.GRU(input_size=embed_size, hidden_size=hidden_size)

    def forward(self, pad_seqs, seq_lengths, hidden):
        """
        Args:
          pad_seqs of shape (max_seq_length, batch_size): Padded source sequences.
          seq_lengths: List of sequence lengths.
          hidden of shape (1, batch_size, hidden_size): Initial states of the GRU.

        Returns:
          outputs of shape (max_seq_length, batch_size, hidden_size): Padded outputs of GRU at every step.
          hidden of shape (1, batch_size, hidden_size): Updated states of the GRU.
        """
        #First we need to apply de embedding layer to our pad_seqs, because otherwise, de function pack_pad_sequence will not work at all. 
        embedded = self.embedding(pad_seqs)#[seq_len,batch_size,embedding_size]
        packed = pack_padded_sequence(embedded,seq_lengths,batch_first=False,enforce_sorted=True)
        outputs_packed, hidden = self.gru(packed,hidden)
        outputs,_ = pad_packed_sequence(outputs_packed,batch_first=False,padding_value=PADDING_VALUE)
        return outputs ,hidden

            

    def init_hidden(self, batch_size=1):
        return torch.zeros(1, batch_size, self.hidden_size)

In [46]:
def test_Encoder_shapes():
    hidden_size = 3
    encoder = Encoder(src_dictionary_size=5, embed_size=10, hidden_size=hidden_size)

    max_seq_length = 4
    batch_size = 2
    hidden = encoder.init_hidden(batch_size=batch_size)
    pad_seqs = torch.tensor([
        [        1,             2],
        [        2,     EOS_token],
        [        3, PADDING_VALUE],
        [EOS_token, PADDING_VALUE]
    ])

    outputs, new_hidden = encoder.forward(pad_seqs=pad_seqs, seq_lengths=[4, 2], hidden=hidden)
    assert outputs.shape == torch.Size([4, batch_size, hidden_size]), f"Bad outputs.shape: {outputs.shape}"
    assert new_hidden.shape == torch.Size([1, batch_size, hidden_size]), f"Bad new_hidden.shape: {new_hidden.shape}"
    print('Success')

test_Encoder_shapes()

Success


In [ ]:
# This cell tests Encoder

## Decoder

The decoder takes as input the representation computed by the encoder and transforms it into a sentence in the target language. The computational graph of the decoder is shown below:

<img src="seq2seq_decoder.png" width=500 align="top">

* $z_0$ is the output of the encoder, that is $z_0 = h_5$, thus `hidden_size` of the decoder should be the same as `hidden_size` of the encoder.
* $y_{i}$ are the log-probabilities of the words in the target language, the dimensionality of $y_{i}$ is the size of the target dictionary.
* $z_{i}$ is mapped to $y_{i}$ using a linear layer `self.out` followed by `F.log_softmax` (because we use `nn.NLLLoss` loss for training).
* Each cell of the decoder is a GRU, it receives as inputs the previous state $z_{i-1}$ and relu of the **embedding** of the previous word. Thus, you need to embed the words of the target language as well. The previous word is taken as the word with the maximum log-probability.

Note that the decoder outputs a word at every step and the same word is used as the input to the recurrent unit at the next step. At the beginning of decoding, the previous word input is fed with a special word SOS which stands for "start of a sentence". During training, we know the target sentence for decoding, therefore we can feed the correct words $y_i$ as inputs to the recurrent unit.

There is one extra thing that it is wise to take care of. When the target sentence is fed to the decoder during training, the decoder learns to generate only the next word (this scenario is called "teacher forcing"). At test time, the decoder works differently: it generates the whole sequence using its own predictions as inputs at each step. Therefore, it makes sense to train the decoder to produce full sentences. In order to do that, we will alternate between two modes during training:
* "teacher forcing": the decoder is fed with the words in the target sequence
* no "teacher forcing": the decoder generates the output sequence using its own predictions. In this case, we will generate sequences of the same length as the length of the longest sequence in `pad_tgt_seqs` (if `pad_tgt_seqs` is not `None`) or of length `MAX_LENGTH` (if `pad_tgt_seqs` is `None`).

You need to implement the decoder which has the structure shown in the figure above.

Notes:
* `SOS_token` is imported at the beginning of the notebook.
* **Running this code on GPU sometimes fails producing a CUDA error (if you know the reason, please let us know).** If this happens to you, please train the model on CPU.

In [47]:
class Decoder(nn.Module):
    def __init__(self, tgt_dictionary_size, embed_size, hidden_size):
        """
        Args:
          tgt_dictionary_size: The number of words in the target dictionary.
          embed_size: The number of dimensions in the word embeddings.
          hidden_size: The number of features in the hidden state.
        """
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        self.tgt_dictionary_size = tgt_dictionary_size

        self.embedding = nn.Embedding(tgt_dictionary_size, embed_size)
        self.gru = nn.GRU(input_size=embed_size, hidden_size=hidden_size)
        self.out = nn.Linear(hidden_size, tgt_dictionary_size)

    def forward(self, hidden, pad_tgt_seqs=None, teacher_forcing=False):
        """
        Args:
          hidden of shape (1, batch_size, hidden_size): States of the GRU.
          pad_tgt_seqs of shape (max_out_seq_length, batch_size): Tensor of words (word indices) of the
              target sentence. If None, the output sequence is generated by feeding the decoder's outputs
              (teacher_forcing has to be False).
          teacher_forcing (bool): Whether to use teacher forcing or not.

        Returns:
          outputs of shape (max_out_seq_length, batch_size, tgt_dictionary_size): Tensor of log-probabilities
              of words in the target language.
          hidden of shape (1, batch_size, hidden_size): New states of the GRU.

        Note: Do not forget to transfer tensors that you may want to create in this function to the device
        specified by `hidden.device`.
        """
        device = hidden.device
        batch_size = hidden.size(1)
        if pad_tgt_seqs is not None:
            max_len = pad_tgt_seqs.size(0)
        else:
            max_len = MAX_LENGTH 
        outputs = torch.empty(max_len, batch_size, self.tgt_dictionary_size, device=device)
        prev_tokens = torch.full((batch_size,), SOS_token, dtype=torch.long, device=device)

        h = hidden 

        for t in range(max_len):
            emb_t = self.embedding(prev_tokens)        
            emb_t = F.relu(emb_t)                  
            emb_t = emb_t.unsqueeze(0)                  
            out_t, h = self.gru(emb_t, h)              
            out_t = out_t.squeeze(0)                    
            logits_t = self.out(out_t)                   
            log_probs_t = F.log_softmax(logits_t, dim=1)
            outputs[t] = log_probs_t
            if teacher_forcing and pad_tgt_seqs is not None:
                if t + 1 < max_len:
                    prev_tokens = pad_tgt_seqs[t]
                else:
                    prev_tokens = pad_tgt_seqs[t]
            else:
                prev_tokens = log_probs_t.argmax(dim=1)

        return outputs, h

In [48]:
def test_Decoder_shapes():
    hidden_size = 2
    tgt_dictionary_size = 5
    test_decoder = Decoder(tgt_dictionary_size, embed_size=10, hidden_size=hidden_size)

    max_seq_length = 4
    batch_size = 2
    pad_tgt_seqs = torch.tensor([
        [        1,             2],
        [        2,     EOS_token],
        [        3, PADDING_VALUE],
        [EOS_token, PADDING_VALUE]
    ])  # [max_seq_length, batch_size]

    hidden = torch.zeros(1, batch_size, hidden_size)
    outputs, new_hidden = test_decoder.forward(hidden, pad_tgt_seqs, teacher_forcing=False)

    assert outputs.size(0) <= 4, f"Too long output sequence: outputs.size(0)={outputs.size(0)}"
    assert outputs.shape[1:] == torch.Size([batch_size, tgt_dictionary_size]), \
        f"Bad outputs.shape[1:]={outputs.shape[1:]}"
    assert new_hidden.shape == torch.Size([1, batch_size, hidden_size]), f"Bad new_hidden.shape={new_hidden.shape}"

    outputs, new_hidden = test_decoder.forward(hidden, pad_tgt_seqs, teacher_forcing=True)
    assert outputs.shape == torch.Size([4, batch_size, tgt_dictionary_size]), \
        f"Bad shape outputs.shape={outputs.shape}"
    assert new_hidden.shape == torch.Size([1, batch_size, hidden_size]), f"Bad new_hidden.shape={new_hidden.shape}"

    # Generation mode
    outputs, new_hidden = test_decoder.forward(hidden, None, teacher_forcing=False)
    assert outputs.shape[1:] == torch.Size([batch_size, tgt_dictionary_size]), \
        f"Bad outputs.shape[1:]={outputs.shape[1:]}"
    assert new_hidden.shape == torch.Size([1, batch_size, hidden_size]), f"Bad new_hidden.shape={new_hidden.shape}"

    print('Success')

test_Decoder_shapes()

Success


In [ ]:
# This cell tests Decoder

In [ ]:
# This cell tests Decoder

In [ ]:
# This cell tests Decoder

## Training of sequence-to-sequence model using mini-batches

Now we are going to train the sequence-to-sequence model on the toy translation dataset.

In [49]:
# Create the seq2seq model
hidden_size = embed_size = 256
encoder = Encoder(trainset.input_lang.n_words, embed_size, hidden_size).to(device)
decoder = Decoder(trainset.output_lang.n_words, embed_size, hidden_size).to(device)

In [50]:
teacher_forcing_ratio = 0.5

Implement the training loop in the cell below. In the training loop, we first encode source sequences using the encoder, then we decode the encoded state using the decoder. The decoder outputs log-probabilities of words in the target language. We need to use these log-probabilities and the indexes of the words in the target sequences to compute the loss.

Recommended hyperparameters:
- Encoder optimizer: Adam with learning rate 0.001
- Decoder optimizer: Adam with learning rate 0.001
- Number of epochs: 30
- Toggle `teacher_forcing` on and off (for each mini-batch) according to the `teacher_forcing_ratio` specified above.

Hints:
- Training should proceed relatively fast.
- If you do well, the training loss should reach 0.1 in 30 epochs.
- **Important:** When computing the loss, you need to ignore the padded values. This can easily be done by using argument `ignore_index` of function [`nll_loss`](
https://pytorch.org/docs/stable/nn.functional.html#torch.nn.functional.nll_loss).

In [53]:
if not skip_training:
    encoder.train()
    decoder.train()

    enc_opt = optim.Adam(encoder.parameters(), lr=0.001)
    dec_opt = optim.Adam(decoder.parameters(), lr=0.001)

    num_epochs = 30

    for epoch in range(1, num_epochs + 1):
        for batch in trainloader:
            pad_src_seqs, src_seq_lengths, pad_tgt_seqs = batch
            pad_src_seqs = pad_src_seqs.to(device, non_blocking=True)
            pad_tgt_seqs = pad_tgt_seqs.to(device, non_blocking=True)

            B = pad_src_seqs.size(1)

            enc_opt.zero_grad()
            dec_opt.zero_grad()
            hidden0 = torch.zeros(1, B, hidden_size, device=device)
            enc_ret = encoder(pad_src_seqs, src_seq_lengths, hidden0)
            enc_out, hidden = enc_ret
            use_tf = (random.random() < teacher_forcing_ratio)
            outputs, hidden = decoder(hidden, pad_tgt_seqs=pad_tgt_seqs, teacher_forcing=use_tf)
            Ttgt, B, V = outputs.size()
            loss = F.nll_loss(
                outputs.view(Ttgt * B, V), #POrque la nll loss espera un tensor de la forma (N,C) y nostros tenemos (T,B,V), por lo cual tenemos que multiplicar los primeros dos puntos. 
                pad_tgt_seqs.view(Ttgt * B),
                ignore_index=PADDING_VALUE
            )
            loss.backward()
            enc_opt.step()
            dec_opt.step()
            print(f"Epoch {epoch:02d}/{num_epochs} - train NLL: {loss.item()}")

c:\Users\guzma\4 IMAT\PRIMER CUATRI\Machines Learning and Perception\Practica_5_RNNs\myenv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 01/30 - train NLL: 7.986486911773682
Epoch 01/30 - train NLL: 7.756248950958252
Epoch 01/30 - train NLL: 7.493960380554199
Epoch 01/30 - train NLL: 7.197259902954102
Epoch 01/30 - train NLL: 6.860788345336914
Epoch 01/30 - train NLL: 6.508228778839111
Epoch 01/30 - train NLL: 6.077762126922607
Epoch 01/30 - train NLL: 5.725608825683594
Epoch 01/30 - train NLL: 5.357430458068848
Epoch 01/30 - train NLL: 5.033011436462402
Epoch 01/30 - train NLL: 4.994806289672852
Epoch 01/30 - train NLL: 4.7027435302734375
Epoch 01/30 - train NLL: 4.5715556144714355
Epoch 01/30 - train NLL: 4.621424198150635
Epoch 01/30 - train NLL: 4.291578769683838
Epoch 01/30 - train NLL: 4.2451558113098145
Epoch 01/30 - train NLL: 4.178020477294922
Epoch 01/30 - train NLL: 4.13682222366333
Epoch 01/30 - train NLL: 4.1236443519592285
Epoch 01/30 - train NLL: 4.112663269042969
Epoch 01/30 - train NLL: 4.097729206085205
Epoch 01/30 - train NLL: 4.230850696563721
Epoch 01/30 - train NLL: 4.433187961578369
Epoch 01

In [54]:
# Save the model to disk (the pth-files will be submitted automatically together with your notebook)
# Set confirm=False if you do not want to be asked for confirmation before saving.
if not skip_training:
    tools.save_model(encoder, '1_rnn_encoder.pth', confirm=True)
    tools.save_model(decoder, '1_rnn_decoder.pth', confirm=True)

Model saved to 1_rnn_encoder.pth.
Model saved to 1_rnn_decoder.pth.


In [56]:
if skip_training:
    hidden_size = 256
    encoder = Encoder(trainset.input_lang.n_words, embed_size, hidden_size)
    tools.load_model(encoder, '1_rnn_encoder.pth', device)
    
    decoder = Decoder(trainset.output_lang.n_words, embed_size, hidden_size)
    tools.load_model(decoder, '1_rnn_decoder.pth', device)

Model loaded from 1_rnn_encoder.pth.
Model loaded from 1_rnn_decoder.pth.


In [ ]:
# This cell tests training accuracy

## Evaluation

Next we need to implement a function that converts input sequences to output sequences using the trained sequence-to-sequence model.

Notes:
* Since we do not need to compute the gradients in the evaluation phase, we can speed up the computations by using the statement `with torch.no_grad():`.
* Please transfer the tensors to `device` inside this function.

In [57]:
def translate(encoder, decoder, pad_src_seqs, src_seq_lengths):
    """Translate sequences from the source language to the target language using the trained model.
    
    Args:
      encoder (Encoder): Trained encoder.
      decoder (Decoder): Trained decoder.
      pad_src_seqs of shape (max_src_seq_length, batch_size): Padded source sequences.
      src_seq_lengths: List of source sequence lengths.
    
    Returns:
      out_seqs of shape (MAX_LENGTH, batch_size): LongTensor of word indices of the output sequences.
    """
    # YOUR CODE HERE
    device = next(encoder.parameters()).device
    pad_src_seqs = pad_src_seqs.to(device)
    hidden_size_enc = getattr(encoder, 'hidden_size', None)
    if hidden_size_enc is None:
        hidden_size_enc = getattr(getattr(encoder, 'gru', None), 'hidden_size', None)
    B = pad_src_seqs.size(1)

    with torch.no_grad():
        encoder.eval()
        decoder.eval()
        hidden0 = torch.zeros(1, B, hidden_size_enc, device=device)
        enc_ret = encoder(pad_src_seqs, src_seq_lengths, hidden0)
        _,hidden = enc_ret
        outputs, _ = decoder(hidden, pad_tgt_seqs=None, teacher_forcing=False)
        out_seqs = outputs.argmax(dim=2)

    return out_seqs

In [58]:
def test_translate_shapes():
    pad_src_seqs = torch.tensor([
        [1, 2],
        [2, 3],
        [3, 0],
        [4, 0]
    ])

    out_seqs = translate(encoder, decoder, pad_src_seqs, src_seq_lengths=[4, 2])
    assert out_seqs.shape == torch.Size([MAX_LENGTH, 2]), f"Wrong out_seqs.shape: {out_seqs.shape}"
    print('Success')

test_translate_shapes()

Success


Let us now translate a few sentences from the training set and print the source, target, and produced output.

If you trained the model well enough, the model should memorize the training data well.

In [59]:
def seq_to_tokens(seq, lang):
    'Convert a sequence of word indices into a list of words (strings).'
    sentence = []
    for i in seq:
        if i == EOS_token:
            break
        sentence.append(lang.index2word[i.item()])
    return(sentence)

def seq_to_string(seq, lang):
    'Convert a sequence of word indices into a sentence string.'
    return(' '.join(seq_to_tokens(seq, lang)))

In [60]:
# Translate a few sentences from the training set
print('Translate training data:')
print('-----------------------------')
pad_src_seqs, src_seq_lengths, pad_tgt_seqs = next(iter(trainloader))
out_seqs = translate(encoder, decoder, pad_src_seqs, src_seq_lengths)

for i in range(5):
    print('SRC:', seq_to_string(pad_src_seqs[:,i], trainset.input_lang))
    print('TGT:', seq_to_string(pad_tgt_seqs[:,i], trainset.output_lang))
    print('OUT:', seq_to_string(out_seqs[:,i], trainset.output_lang))
    print('')

Translate training data:
-----------------------------
SRC: j ai du travail par dessus la tete .
TGT: i am up to my neck in work .
OUT: i am up to my neck in work .

SRC: je n en peux plus de l anglais .
TGT: i m sick of english .
OUT: i m fed up with english .

SRC: nous sommes sur le chemin de la maison .
TGT: we re on our way home .
OUT: we re on our way home .

SRC: il a un peu plus de quarante ans .
TGT: he is a little over forty .
OUT: he is a little over forty .

SRC: j ai l habitude de me lever tot .
TGT: i m used to getting up early .
OUT: i m used to getting up early .



c:\Users\guzma\4 IMAT\PRIMER CUATRI\Machines Learning and Perception\Practica_5_RNNs\myenv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Now we translate random sentences from the test set. A well-trained model should output sentences that look similar to the target ones. The mistakes are usually done for words that were rare in the training set.

In [61]:
testset = TranslationDataset(data_dir, train=False)
testloader = DataLoader(dataset=testset, batch_size=64, shuffle=True, collate_fn=collate)

In [62]:
print('Translate test data:')
print('-----------------------------')
pad_src_seqs, src_seq_lengths, pad_tgt_seqs = next(iter(testloader))
out_seqs = translate(encoder, decoder, pad_src_seqs, src_seq_lengths)

for i in range(5):
    print('SRC:', seq_to_string(pad_src_seqs[:,i], testset.input_lang))
    print('TGT:', seq_to_string(pad_tgt_seqs[:,i], testset.output_lang))
    print('OUT:', seq_to_string(out_seqs[:,i], testset.output_lang))
    print('')

Translate test data:
-----------------------------
SRC: il est toujours en retard a l ecole .
TGT: he is always late for school .
OUT: he s always late for school .

SRC: je suis desole de te deranger si souvent .
TGT: i m sorry to bother you so often .
OUT: i m sorry to bother you so often .

SRC: il est sur le point d y aller .
TGT: he s about to go .
OUT: he is looking to go .

SRC: je suis desole je ne te reconnais pas .
TGT: i m sorry i don t recognize you .
OUT: i m sorry i don t recognize you .

SRC: je ne fais pas partie de leur groupe .
TGT: i m not one of them .
OUT: i m not one of them .



## Compute BLEU score

Let us now compute the [BLEU score](https://en.wikipedia.org/wiki/BLEU) for the translations produced by our model. We can use the HuggingFace evaluate function [bleu](https://huggingface.co/spaces/evaluate-metric/bleu) for that.

* **Our model should achieve a BLEU score of 95 on the training set.**
* The BLEU score on the test set can be significantly smaller (about 46).

The model can severly overfit to the training set and we do not cope with the overfitting problem in this exercise.

In [63]:
bleu = evaluate.load("bleu")

def convert_to_sentences(word_lists):
    return [' '.join(words) for words in word_lists]
def wrap_sentences(token_lists):
    return [[' '.join(tokens[0])] for tokens in token_lists]

In [64]:
# Create translations for the training set
candidate_corpus = []
references_corpus = []
for pad_src_seqs, src_seq_lengths, pad_tgt_seqs in trainloader:
    out_seqs = translate(encoder, decoder, pad_src_seqs, src_seq_lengths)
    candidate_corpus.extend([seq_to_tokens(seq, trainset.output_lang) for seq in out_seqs.T])
    references_corpus.extend([[seq_to_tokens(seq, trainset.output_lang)] for seq in pad_tgt_seqs.T])

# Compute BLEU for translations
results = bleu.compute(predictions=convert_to_sentences(candidate_corpus), references=wrap_sentences(references_corpus))
score = results["bleu"]
print(f'BLEU score on training data: {score*100}')
assert score*100 > 90, "The BLEU score is too low."

BLEU score on training data: 96.93950218653185


In [65]:
# Create translations for the test set
candidate_corpus = []
references_corpus = []
for pad_src_seqs, src_seq_lengths, pad_tgt_seqs in testloader:
    out_seqs = translate(encoder, decoder, pad_src_seqs, src_seq_lengths)
    candidate_corpus.extend([seq_to_tokens(seq, testset.output_lang) for seq in out_seqs.T])
    references_corpus.extend([[seq_to_tokens(seq, testset.output_lang)] for seq in pad_tgt_seqs.T])

# Compute BLEU for translations
results = bleu.compute(predictions=convert_to_sentences(candidate_corpus), references=wrap_sentences(references_corpus))
score = results["bleu"]
print(f'BLEU score on test data: {score*100}')

BLEU score on test data: 48.57724368470149


<div class="alert alert-block alert-info">
<b>Conclusion</b>
</div>

In this notebook:
* We learned how recurrent neural networks can be used to build a sequence-to-sequence model.
* We trained a sequence-to-sequence model for statistical machine translation.